# 3 — First operators: `keep`, `drop`, `rename` <img style="float: right;" src="images/VTL_23.png" width="100" height="80">

**Prerequisites:** [1.1-trevas-intro.ipynb](1.1-trevas-intro.ipynb) · [1.2-vtl-intro.ipynb](1.2-vtl-intro.ipynb) · [2-core-rules.ipynb](2-core-rules.ipynb) (structured `grades` data set).

**Goals**
- Project columns with `keep` and `drop`
- Rename components with `rename`
- Chain clause operators
- Export the result with `writeCSV`

These three operators are **clause operators** in VTL 2.1 — they transform the **structure** of a data set while leaving identifiers intact.


## Setup — load and structure the training data <img style="float: right;" src="images/VTL_27.png" width="30" height="80">

Re-run this cell if you open this notebook in a fresh session.


In [ ]:
grades_raw <- loadCSV("data/students_grades.csv?delimiter=,");

grades := grades_raw
  [calc identifier Student_Id := Student_Id,
        identifier Subject := Subject,
        identifier Semester := Semester,
        measure Grade := Grade,
        attribute School := School,
        attribute Teacher := Teacher];

## Syntax reminder

| Operator | Syntax | Effect on measures/attributes |
|---|---|---|
| **keep** | `ds [ keep Comp1, Comp2 ]` | Keep listed components, **drop all other** measures and attributes |
| **drop** | `ds [ drop Comp1, Comp2 ]` | Remove listed components, **keep all others** |
| **rename** | `ds [ rename Old to New ]` | Change component names (values unchanged) |

**Common rule:** `keep` and `drop` cannot target **identifiers** — identifiers are always kept.

Think of `keep` / `drop` as **column projection** on dependent variables; identifiers define the grain of the table.


## `keep` — keep only selected measures or attributes

Keep the **Grade** measure only — identifiers `Student_Id`, `Subject`, `Semester` remain.


In [ ]:
grades_scores := grades [ keep Grade ];
scores_preview := show(grades_scores);

Keep only descriptive attributes (School, Teacher). All measures except those listed are removed — here `Grade` is dropped from the result.


In [ ]:
grades_context := grades [ keep School, Teacher ];
context_meta := showMetadata(grades_context);

## `drop` — remove selected components

Remove contextual attributes to simplify the table.


In [ ]:
grades_slim := grades [ drop School, Teacher ];
slim_preview := show(grades_slim);

Drop the measure — rarely useful alone, but illustrates that identifiers survive.


In [ ]:
ids_only := grades [ drop Grade ];
ids_meta := showMetadata(ids_only);

## `rename` — change component names

`rename` changes **names only**, not values. The new name must not collide with an existing component.

Rename the measure for a downstream publication step:


In [ ]:
grades_renamed := grades [ rename Grade to Exam_score ];
renamed_meta := showMetadata(grades_renamed);
renamed_preview := show(grades_renamed);

Rename an attribute:

```vtl
ds [ rename School to School_name ]
```

Multiple renames in one clause:

```vtl
ds [ rename Grade to Exam_score, School to School_name ]
```


## Chaining clauses

Operators apply **left to right**. Example: drop teacher, then rename school.


In [ ]:
grades_clean := grades
  [ drop Teacher ]
  [ rename School to School_name ];

clean_meta := showMetadata(grades_clean);
clean_preview := show(grades_clean);

## Export result — `writeCSV`

After transforming a data set, persist it with Trevas `writeCSV`. Spark writes a **directory** (one or more part files), not a single file:

```vtl
written := writeCSV("./output/grades_clean", grades_clean);
```

Open the `output/grades_clean/` folder in the Jupyter file browser to download the CSV parts.


In [ ]:
written := writeCSV("./output/grades_clean", grades_clean);

## `keep` vs `drop` — when to use which?

| Situation | Prefer |
|---|---|
| Few columns to **keep** | `keep Col1, Col2` |
| Few columns to **remove** | `drop Col3, Col4` |
| Identifiers | Never listed — always preserved |

## Practice ideas

1. Keep only `Grade` and `School`.
2. Try `grades [ drop Semester ]` — why does it fail? (`Semester` is an identifier; `drop` only applies to measures and attributes.)
3. Rename attributes to snake_case names.

---

Back to [2-core-rules.ipynb](2-core-rules.ipynb) · [1.2-vtl-intro.ipynb](1.2-vtl-intro.ipynb) · [1.1-trevas-intro.ipynb](1.1-trevas-intro.ipynb)
